In [ ]:
%%! 
pip install -r requirements.txt

In [83]:
import pandas as pd

systembook = pd.read_csv("data/03_Library Systembook.csv")
systemCustomers = pd.read_csv("data/03_Library SystemCustomers.csv")

systembook  # shows the first 5 rows

,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
0,1.0,Catcher in the Rye,"""20/02/2023""",25/02/2023,2 weeks,1.0
1,2.0,Lord of the rings the two towers,"""24/03/2023""",21/03/2023,2 weeks,2.0
2,3.0,Lord of the rings the return of the kind,"""29/03/2023""",25/03/2023,2 weeks,3.0
3,4.0,The hobbit,"""02/04/2023""",25/03/2023,2 weeks,4.0
4,5.0,Dune,"""02/04/2023""",25/03/2023,2 weeks,5.0
...,...,...,...,...,...,...
109,NaN,NaN,NaN,NaN,NaN,NaN
110,NaN,NaN,NaN,NaN,NaN,NaN
111,NaN,NaN,NaN,NaN,NaN,NaN
112,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
systemCustomers  # shows the first 5 rows

,Customer ID,Customer Name
0,1.0,Jane Doe
1,2.0,John Smith
2,3.0,Dan Reeves
3,NaN,NaN
4,5.0,William Holden
5,6.0,Jaztyn Forest
6,7.0,Jackie Irving
7,8.0,Matthew Stirling
8,9.0,Emory Ted


In [84]:
# Remove empty or NAN rows 

systembook = systembook.dropna(how="all")
systemCustomers = systemCustomers.dropna(how="all")

# Check for empty book names

systembook["Valid Rows"] = "Pass"
systembook["Valid Rows"] = systembook["Valid Rows"].mask(systembook["Books"].isna(), "Fail - invalid book name")

# Strip the '"" from dates

systembook["Book checkout"] = systembook["Book checkout"].str.strip('"')

# Convert IDs to int 

systembook["Customer ID"] = systembook["Customer ID"].astype("Int64")
systembook["Id"] = systembook["Id"].astype("Int64")
systemCustomers["Customer ID"] = systemCustomers["Customer ID"].astype("Int64")

# Convert weeks to days 

systembook["Days allowed to borrow"] = systembook["Days allowed to borrow"].str.extract(r"(\d+)", expand=False).astype("Int64") * 7

# Convert to datetime + remove invalid dates 
for c in ["Book checkout", "Book Returned"]:
    systembook[c] = pd.to_datetime(systembook[c], format="%d/%m/%Y", errors="coerce")

# For any weird dates, this covers: 

# Invalid date
systembook["Valid Rows"] = systembook["Valid Rows"].mask(systembook["Book checkout"].isna() | systembook["Book Returned"].isna(), "Fail - invalid date")

# Book checkout >= Book Returned
systembook["Valid Rows"] = systembook["Valid Rows"].mask(systembook["Book checkout"] >= systembook["Book Returned"], "Fail - checkout not before return")

# The checkout date or return is in the future
systembook["Valid Rows"] = systembook["Valid Rows"].mask((systembook["Book checkout"] > pd.Timestamp.today()) | (systembook["Book Returned"] > pd.Timestamp.today()), "Fail - future date")

# Move failed into seperate folder

invalidRows_systembook = systembook[systembook["Valid Rows"] != "Pass"]

# Cleanup failed + remove column

systembook = systembook[systembook["Valid Rows"] == "Pass"]
systembook = systembook.drop(columns=["Valid Rows"])

C:\Users\Admin\AppData\Local\Temp\ipykernel_12828\1943949692.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  systembook["Valid Rows"] = "Pass"
C:\Users\Admin\AppData\Local\Temp\ipykernel_12828\1943949692.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  systembook["Valid Rows"] = systembook["Valid Rows"].mask(systembook["Books"].isna(), "Fail - invalid book name")
C:\Users\Admin\AppData\Local\Temp\ipykernel_12828\1943949692.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sli

In [72]:
invalidRows_systembook

,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID,Valid Rows
1,2,Lord of the rings the two towers,2023-03-24,2023-03-21,14,2,Fail - checkout not before return
2,3,Lord of the rings the return of the kind,2023-03-29,2023-03-25,14,3,Fail - checkout not before return
3,4,The hobbit,2023-04-02,2023-03-25,14,4,Fail - checkout not before return
4,5,Dune,2023-04-02,2023-03-25,14,5,Fail - checkout not before return
6,7,IT,2063-04-10,2023-04-03,14,6,Fail - future date
7,8,Misery,2023-04-15,2023-04-03,14,7,Fail - checkout not before return
16,17,The Bloody Chamber,NaT,2023-06-04,14,3,Fail - invalid date


In [ ]:
systemCustomers

,Customer ID,Customer Name
0,1,Jane Doe
1,2,John Smith
2,3,Dan Reeves
4,5,William Holden
5,6,Jaztyn Forest
6,7,Jackie Irving
7,8,Matthew Stirling
8,9,Emory Ted


In [85]:
import os
os.makedirs("exportData", exist_ok=True)


#Output files 

systembook.to_csv("exportData/systembook.csv", index=False)
systemCustomers.to_csv("exportData/systemCustomers.csv", index=False)
invalidRows_systembook.to_csv("exportData/systembook_invalidRows.csv", index=False)